## Vector Database Demo
#### Vector Search using Azure AI Search Vector Database

In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI

# Load variables from .env in the workspace root
load_dotenv(dotenv_path=Path('.env'))

def required_env(name: str) -> str:
    value = os.getenv(name, '').strip()
    if not value:
        raise ValueError(f"Missing required environment variable: {name}")
    return value

# ---- Azure Search ----
search_client = SearchClient(
    endpoint=required_env('AZURE_SEARCH_ENDPOINT'),
    index_name=required_env('AZURE_SEARCH_INDEX_NAME'),
    credential=AzureKeyCredential(required_env('AZURE_SEARCH_API_KEY'))
)

# ---- Azure OpenAI for Embedding ----
# Model: text-embedding-ada-002
embed_client = AzureOpenAI(
    api_key=required_env('AZURE_OPENAI_API_KEY'),
    azure_endpoint=required_env('AZURE_OPENAI_ENDPOINT'),
    api_version=os.getenv('AZURE_OPENAI_API_VERSION', '2024-02-15-preview').strip() or '2024-02-15-preview',
)
emb_model=os.getenv('AZURE_OPENAI_EMBD_MODEL', 'text-embedding-ada-002').strip() or 'text-embedding-ada-002'


### Perform Vector Search with Azure Search and Azure OpenAI Embeddings
#### Create Embeddings for the Input Query
- Input query in natural language is converted to vectors for search

In [2]:
query = "What is the leave policy?"
#query = "which plan has the lowest deductible"
#query = "What is the vacation policy?"
#query = "What is the policy for remote working?"
#query = "What is the travel policy?"

query_vector = embed_client.embeddings.create(
    model=emb_model, input=query
).data[0].embedding

results = list(search_client.search(
    search_text=query,
    vector_queries=[{
        "kind": "vector",
        "vector": query_vector,
        "fields": "text_vector",
        "k": 3
    }],
    select=["title", "chunk", "text_vector"],
    top=3,
) )

context_chunks = []
context_titles = []

#### Use this output of query vector in Azure Portal
{
  "vectorQueries": [
    {
      "kind": "vector",
      "vector": [0.0123, -0.4567, 0.7890, ...],
      "fields": "text_vector",
      "k": 3
    }
  ],
  "select": "chunk"
}

#### Build the context payload for RAG

In [3]:
context_chunks = []
context_titles = []

print("Query: " + query)
print("-" * 80)

for rank, result in enumerate(results, start=1):
    title = result["title"]
    chunk = result["chunk"]
    vector = result["text_vector"]
    score = result.get("@search.score", 0.0)
    compact_chunk = " ".join(chunk.split())
    preview = compact_chunk[:600] + ("..." if len(compact_chunk) > 600 else "")

    context_chunks.append(chunk)
    context_titles.append(title)

    print(f"Rank: {rank} | Score: {score:.4f} | Characters: {len(chunk):,} | Words: {len(chunk.split()):,}")
    print(f"Title: {title}")
    print(f"Excerpt: {preview}")
    print(f"Vector (5 Ds): {vector[:5]}")
    print("-" * 80)

# Full retrieved text remains available for a later RAG prompt.
rag_context = "\n\n".join(context_chunks)

Query: What is the leave policy?
--------------------------------------------------------------------------------
Rank: 1 | Score: 0.0333 | Characters: 1,983 | Words: 312
Title: Company HR Policy Manual.pdf
Excerpt: Company HR Policy Manual 1. Introduction This document outlines the Human Resources policies applicable to all employees of the organization. These policies are designed to ensure a fair, productive, and compliant workplace. 2. Leave Policy Employees are entitled to 20 days of paid annual leave per calendar year. • Leave can be carried forward up to a maximum of 10 days into the next year. • Any unused leave beyond the carry-forward limit will expire. • Employees are required to apply for leave at least 3 days in advance, except in case of emergencies. • Sick leave is limited to 10 days per ye...
Vector (5 Ds): [0.0007157972, 0.013444824, 0.010118134, -0.04657366, -0.018776733]
--------------------------------------------------------------------------------
Rank: 2 | Score: